# Reading

In [17]:
import geopandas as gpd
import os
from pathlib import Path

In [18]:
parent_path = "C:/Users/quintyn_come/Codex/AIBT102/"

In [19]:
FICHIER_IRIS = (
    Path(parent_path)
    / "scraping_and_preprocessing"
    / "raw_data"
    / "CONTOURS-IRIS_3-0__GPKG_LAMB93_FXX_2025-01-01"
    / "CONTOURS-IRIS"
    / "1_DONNEES_LIVRAISON_2025-06-00117"
    / "CONTOURS-IRIS_3-0_GPKG_LAMB93_FXX-ED2025-01-01"
    / "iris.gpkg"
)

iris_2025 = gpd.read_file(FICHIER_IRIS)

In [20]:
iris_2025

,cleabs,code_insee,nom_commune,iris,code_iris,nom_iris,type_iris,geometry
0,IRIS____0000000670430101,67043,Bischheim,0101,670430101,Annexe,A,"MULTIPOLYGON (((1052346.7 6848413, 1052502.3 6..."
1,IRIS____0000000132020101,13202,Marseille 2e Arrondissement,0101,132020101,Arenc,A,"MULTIPOLYGON (((891859 6248420, 891827 6248420..."
2,IRIS____0000000561850101,56185,Quéven,0101,561850101,B.A.N. Lann Bihoué,A,"MULTIPOLYGON (((219079.3 6760205.2, 219092.3 6..."
3,IRIS____0000000930630101,93063,Romainville,0101,930630101,Bas Pays,A,"MULTIPOLYGON (((657957.7 6866026.2, 658042.5 6..."
4,IRIS____0000000940480101,94048,Marolles-en-Brie,0101,940480101,Bois de Notre-Dame,A,"MULTIPOLYGON (((666452.1 6849977.7, 666654.7 6..."
...,...,...,...,...,...,...,...,...
48507,IRIS____00000002B3640000,2B364,Zuani,0000,2B3640000,Zuani,Z,"MULTIPOLYGON (((1222895.4 6152070.5, 1223071.6..."
48508,IRIS____0000000629050000,62905,Zudausques,0000,629050000,Zudausques,Z,"MULTIPOLYGON (((640862.7 7074791.9, 640963.7 7..."
48509,IRIS____0000000629060000,62906,Zutkerque,0000,629060000,Zutkerque,Z,"MULTIPOLYGON (((634861.6 7083677.7, 634799.2 7..."
48510,IRIS____0000000596680000,59668,Zuydcoote,0000,596680000,Zuydcoote,Z,"MULTIPOLYGON (((665080.1 7107281.1, 664559.9 7..."


In [21]:
iris_2025.columns

Index(['cleabs', 'code_insee', 'nom_commune', 'iris', 'code_iris', 'nom_iris',
       'type_iris', 'geometry'],
      dtype='str')

# Cleaning

In [22]:
iris_2025.isna().sum()

cleabs         0
code_insee     0
nom_commune    0
iris           0
code_iris      0
nom_iris       0
type_iris      0
geometry       0
dtype: int64

In [23]:
print('Number of rows:', iris_2025.shape[0])

Number of rows: 48512


There are no missing values.

Let's extract the IRIS code:

In [24]:
iris_2025["IRIS"] = iris_2025["code_iris"]

In [25]:
iris_2025.head()

,cleabs,code_insee,nom_commune,iris,code_iris,nom_iris,type_iris,geometry,IRIS
0,IRIS____0000000670430101,67043,Bischheim,0101,670430101,Annexe,A,"MULTIPOLYGON (((1052346.7 6848413, 1052502.3 6...",670430101
1,IRIS____0000000132020101,13202,Marseille 2e Arrondissement,0101,132020101,Arenc,A,"MULTIPOLYGON (((891859 6248420, 891827 6248420...",132020101
2,IRIS____0000000561850101,56185,Quéven,0101,561850101,B.A.N. Lann Bihoué,A,"MULTIPOLYGON (((219079.3 6760205.2, 219092.3 6...",561850101
3,IRIS____0000000930630101,93063,Romainville,0101,930630101,Bas Pays,A,"MULTIPOLYGON (((657957.7 6866026.2, 658042.5 6...",930630101
4,IRIS____0000000940480101,94048,Marolles-en-Brie,0101,940480101,Bois de Notre-Dame,A,"MULTIPOLYGON (((666452.1 6849977.7, 666654.7 6...",940480101


# Enriching

Let's add the area of each IRIS:

In [26]:
iris_2025["surface_m2"] = iris_2025.geometry.area

In [27]:
iris_2025['cleabs']

0        IRIS____0000000670430101
1        IRIS____0000000132020101
2        IRIS____0000000561850101
3        IRIS____0000000930630101
4        IRIS____0000000940480101
                   ...           
48507    IRIS____00000002B3640000
48508    IRIS____0000000629050000
48509    IRIS____0000000629060000
48510    IRIS____0000000596680000
48511    IRIS____0000000596690000
Name: cleabs, Length: 48512, dtype: str

In [28]:
colonnes = [
    "IRIS",
    "nom_commune",
    "nom_iris",
    "surface_m2",
]
iris_2025 = iris_2025[colonnes].copy()

In [29]:
iris_2025.columns

Index(['IRIS', 'nom_commune', 'nom_iris', 'surface_m2'], dtype='str')

In [30]:
iris_2025 = iris_2025.rename(columns={
    'nom_commune': 'commune_name',
    'nom_iris': 'iris_name',
})

In [31]:
iris_2025["IRIS"] = iris_2025["IRIS"].astype(str)

# Saving

In [32]:
output_dir = os.path.join(parent_path, "cleaning/cleaned_data")
output_name = "geographical_area.csv"
output_path = os.path.join(output_dir, output_name)
os.makedirs(output_dir, exist_ok=True)
iris_2025.to_csv(
    output_path,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)